# 02 · Blocking and candidate generation

Brute force is impossible: S1 × (S2 ∪ S3) ≈ 2.2M × 10.3M ≈ **2.3 × 10¹³ pairs**. This stage cuts
that to ~10 candidates per S2/S3 record while keeping almost every true match — its recall is the
ceiling for everything downstream.

**Method** (`src/ber/blocking.py`)
1. **Block = country** (lossless, notebook 01).
2. Within a block, **sparse TF-IDF nearest neighbours from every S2/S3 record into S1** on one
   combined vector: char 3-grams of the name ⊕ word 1–2-grams of the address, each l2-normalised,
   so cosine = ½·name-cosine + ½·address-cosine. Top-`C.TOPK` per record. (On the dev sample this
   single channel beat the union of separate name / address / combined channels at equal cost.)
3. N-grams present in more than `RETRIEVAL_MAX_DF` S1 records are dropped from the retrieval
   vectors — no identity signal, most of the sparse-matmul cost.

Only one block's S1 index is held in memory; queries stream in chunks of `C.QUERY_CHUNK_ROWS`.
While streaming, the best and second-best retrieval score per S2/S3 record and per S1 entity are
accumulated — the inputs of the competition features in notebook 03.
Outputs: `candidates/train/{country}.parquet`, `candidates/train_stats.npz`.

In [ ]:
import sys, time, gc
from pathlib import Path

# notebooks/ -> ../src holds the shared, importable pipeline package `ber`
sys.path.insert(0, str(Path.cwd().resolve().parent / "src"))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from ber import config as C, io, pipeline as P

pd.set_option("display.max_colwidth", 80)
pd.set_option("display.width", 200)
C.ensure_dirs()
print(C.describe())
try:
    import psutil
    print(f"RAM: {psutil.virtual_memory().total / 1e9:.1f} GB | CPUs: {psutil.cpu_count()}")
except ImportError:
    pass

> **Infrastructure & cost.** Built to run the *full* dataset on a free-tier-class EC2 instance
> (m7i-flex.large: 2 vCPU, 8 GB RAM + 4 GB swap, 30 GB gp3) — every step streams or chunks
> its data. Checkpoints go to the local EBS volume (which survives stop/start) and small,
> valuable artifacts (model, metadata, submission) are mirrored to S3; big intermediates are
> kept local to stay inside the S3 free tier (5 GB, ~2k PUTs/month). S3→EC2 transfer in the
> same region is free. Re-running a notebook skips stages whose checkpoint exists
> (set `FORCE = True` to recompute). Everything is CPU-bound; no GPU is used.

In [ ]:
FORCE = False
sizes = P.source_sizes("train")
print(sizes, "| countries:", P.countries("train"))
print(f"top-K={C.TOPK} | max_df={P.retrieval_max_df(sizes['s1'])} | min_sim={C.MIN_SIM}")

## 1. Block report (country blocks) and blocking recall

In [ ]:
c1 = pd.read_parquet(io.fetch("clean/train_s1.parquet"), columns=["entity_id", "country"])
cr = pd.concat([pd.read_parquet(io.fetch(f"clean/train_{k}.parquet"), columns=["entity_id", "country"]) for k in ("s2", "s3")],
               ignore_index=True)
blocks = pd.concat([c1.country.value_counts().rename("S1"), cr.country.value_counts().rename("R")], axis=1).fillna(0).astype(int)
blocks["pairs_if_brute_force"] = blocks.S1 * blocks.R
print(blocks)
print(f"\nbrute-force pairs overall : {sizes['s1'] * (sizes['s2'] + sizes['s3']):.3e}")
print(f"pairs after country block : {blocks.pairs_if_brute_force.sum():.3e}")
print(f"mean / median block size  : {blocks.pairs_if_brute_force.mean():.3e} / {blocks.pairs_if_brute_force.median():.3e} pairs")
tp = P.load_true_pairs()
same = (c1.set_index("entity_id")["country"].reindex(tp.source1_entity_id).to_numpy()
        == cr.set_index("entity_id")["country"].reindex(tp.candidate_entity_id).to_numpy())
print(f"\nBLOCKING RECALL (country): {100 * same.mean():.4f}%")
del c1, cr, tp; gc.collect()

## 2. TF-IDF top-K retrieval inside blocks (checkpointed)

In [ ]:
t0 = time.time()
stats = P.run_candidates("train", force=FORCE)
print(f"candidate generation: {time.time() - t0:.0f}s")

## 3. Candidate report — recall is the ceiling for the classifier

In [ ]:
keys, n1 = P.true_pair_keys("train")
hit = np.zeros(len(keys), bool)
rk_true = np.zeros(len(keys), np.int16)
n_pairs, per_s1 = 0, np.zeros(n1, np.int32)
for rel in P.candidate_files("train"):
    c = pd.read_parquet(io.fetch(rel))
    k = c.r_idx.to_numpy().astype(np.int64) * n1 + c.s1_idx.to_numpy()
    order = np.argsort(k); ks = k[order]
    pos = np.searchsorted(ks, keys); pos[pos >= len(ks)] = 0
    m = ks[pos] == keys
    hit |= m
    rk_true[m] = c.rk.to_numpy()[order][pos[m]]
    n_pairs += len(c)
    per_s1 += np.bincount(c.s1_idx.to_numpy(), minlength=n1).astype(np.int32)
    del c, k, order, ks; gc.collect()
n_r = sizes["s2"] + sizes["s3"]
print(f"CANDIDATE RECALL: {hit.mean():.5f}  ({hit.sum():,} of {len(keys):,} true pairs)")
print(f"candidate pairs : {n_pairs:,} ({n_pairs / n_r:.2f} per S2/S3 record)")
print(f"candidates per S1: mean {per_s1.mean():.1f} | median {np.median(per_s1):.0f} | p99 {np.percentile(per_s1, 99):.0f} | zero {100 * (per_s1 == 0).mean():.2f}%")
print(f"reduction ratio : {1 - n_pairs / (n1 * n_r):.10f}")

### Recall vs. K

Each extra neighbour costs feature/scoring time; this shows what a smaller K would keep
(computed from the stored rank of each true pair — no re-retrieval needed).

In [ ]:
pd.DataFrame({"K": np.arange(1, C.TOPK + 1),
              "recall": [((rk_true >= 1) & (rk_true <= k)).mean() for k in range(1, C.TOPK + 1)]}).round(5)

### Examples of missed true matches

In [ ]:
s1_ids, r_ids = P.entity_ids("train")
miss = np.flatnonzero(~hit)
rng = np.random.default_rng(0)
cols = ["entity_id", "name_core", "addr_clean"]
s1c = pd.read_parquet(io.fetch("clean/train_s1.parquet"), columns=cols).set_index("entity_id")
rc = pd.concat([pd.read_parquet(io.fetch(f"clean/train_{k}.parquet"), columns=cols) for k in ("s2", "s3")]).set_index("entity_id")
for kk in rng.choice(miss, size=min(12, len(miss)), replace=False):
    a, b = s1_ids.iloc[kk % n1], r_ids.iloc[kk // n1]
    print(f"S1: {s1c.loc[a].name_core!r:38} | {s1c.loc[a].addr_clean!r}\n{b[:2]}: {rc.loc[b].name_core!r:38} | {rc.loc[b].addr_clean!r}\n")
del s1c, rc; gc.collect()

## 4. Learned candidate pruning (cascade stage 2b) — the final candidate set

The challenge ranks teams with **smaller candidate sets per S1 entity** higher, and
`candidate_pairs.tsv` must be exactly the set the matcher scores. A small LightGBM **pruner** uses
only retrieval-stage signals (retrieval score and its name / address parts, rank, competition per
record and per S1, near-ties, name/address frequency — no string comparisons, so it is cheap) to
score every retrieved pair; pairs with `q < PRUNE_Q` are dropped. It is trained on a slice of
*train-role* entities only (never on validation entities).

Measured on the full India block: 70.4 → 9.3 candidates per S1 at `PRUNE_Q=0.001`
(F0.5 −0.0001), 7.5 at 0.003 (−0.0002), 6.0 at 0.01 (−0.0005). The survivors are also the only
pairs that get the expensive features, which cuts feature / scoring time ~8×.

In [ ]:
t0 = time.time()
summary = P.run_prune("train", force=FORCE)
print(summary, f"({time.time() - t0:.0f}s)")
keys, n1 = P.true_pair_keys("train")
for label, pruned in (("retrieved (top-K)", False), ("after pruning", True)):
    got, npairs = 0, 0
    for rel in P.candidate_files("train", pruned=pruned):
        c = pd.read_parquet(io.fetch(rel), columns=["s1_idx", "r_idx"])
        k = c.r_idx.to_numpy().astype(np.int64) * n1 + c.s1_idx.to_numpy()
        got += int(np.isin(keys, k).sum()); npairs += len(c)
        del c, k; gc.collect()
    print(f"{label:18s}: {npairs:,} pairs = {npairs / n1:.1f} per S1 entity | candidate recall {got / len(keys):.5f}")

Remaining misses are mostly records with an **empty address and a generic or heavily garbled
name** — even a human could not link them confidently, and a precision-weighted metric would
reject most of them anyway.